### Universidade Federal da Bahia
### Escola Politécnica
### Programa de Pós-Graduação em Engenharia Industrial
### Discente: Ghabriel Anton Gomes de Sá
### Orientadores: Marcelo Embiruçu e Cristiano Fontes

<center><h1>UMA ABORDAGEM COMBINANDO UMA ESTIMATIVA DE PESOS INICIAIS BASEADA EM LINEARIZAÇÃO E ALGORITMO CONSTRUTIVISTA PARA REDES NEURAIS DE PROPAGAÇÃO DIRETA COM UMA ÚNICA CAMADA OCULTA APLICADAS À IDENTIFIÇÃO DE MODELOS COM MÚLTIPLAS ENTRADAS E ÚNICA SAÍDA EM PROBLEMAS DE REGRESSÃO COM ANÁLISE DE GANHOS</center></h1>

#### <p style='text-align: justify;'>O Método para a obtenção dos pesos iniciais propõe que: (i) seja realizada a linearização, via série de Taylor, do modelo não-linear a ser utilizado para a modelagem do conjunto de dados (o modelo consiste em uma rede neural com p entradas e um neurônio na camada de saída), onde o ponto de equilíbrio para a linearização é a média das suas respectivas variáveis (x,y); (ii) seja realizada a Regressão Linear Múltipla, via Método dos Mínimos Quadrados, do conjunto de dados a ser modelado; (iii) através da comparação matemática entre (i) e (ii), obtenha-se os pesos que conectam os neurônios de entrada ao primeiro neurônio oculto da rede.
#### <p style='text-align: justify;'>Em seguida, a rede continua a aumentar o número de neurônios ocultos a partir de uma abordagem construtivista. Toda a rede é treinada a cada adição de um novo neurônio oculto, sendo que os pesos obtidos no final de um treinamento são utilizados como pesos iniciais para o treinamento subsequente. Os pesos iniciais do neurônio oculto a ser adicionado são obtidos randomicamente, via método de Xavier; biases são inicializados com valor nulo. Comparou-se o método proposto (WILCA-R) com outros dois métodos (RIXM e ELM).

## Importação de pacotes

In [ ]:
# Manipulação de dados:
import pandas as pd
from sklearn import preprocessing
from sklearn.utils import shuffle

# Resolução de sistema de eq. não-lineares:
import statsmodels.api as sm
from scipy.optimize import fsolve

# Álgebra linear:
import numpy as np
import math
from scipy.linalg import pinv # usado no ELM

# Cálculo do R2:
from sklearn.metrics import r2_score

# Contagem de tempo:
import time

# Criação e plotagem de gráficos:
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

## Fixação de seed

In [ ]:
np.random.seed(0)

## Importação do banco de dados
<p style='text-align: justify;'>Para que a importação do banco de dados seja feita da maneira correta, é necessário que o banco de dados a ser importado esteja de acordo com o padrão apresentado a seguir e também que ele esteja localizado na mesma pasta que esse código. A princípio, esse código está implementado em um arquivo .ipynb.</p>

O banco de dados deve apresentar, obrigatoriamente, as seguintes características:

-Deve ser um arquivo do tipo .csv;

-Os dados devem estar delimitados por ponto e vírgula (;), ou seja, devem estar presentes em diferentes células da planilha Excel;

-As variáveis de entradas são todas as colunas, exceto a última (que trata-se da variável de saída);

-Não deveve possuir cabeçalho (linha com nome dos atributos).

In [ ]:
# Pedindo informações acerca do conjunto de dados (nome do arquivo e se ele possui cabeçalho ou não):
filename = input("Digite o nome do arquivo .csv que contém o banco de dados:")

# Abrindo conjunto de dados:
data = pd.read_csv(filename+'.csv', header=None, sep=';')

## Pré-processamento de dados

#### • Obtendo primeiras informações sobre o conjunto de dados:

In [ ]:
# Mostrando cinco primeiras linhas de dados:
data.head()

In [ ]:
# Mostrando informações sobre os atributos:
data.info()

In [ ]:
# Descrição estatística dos dados:
data.describe()

#### • Lidando com os dados faltantes:

In [ ]:
# Verificando a existência de dados faltantes:
data.isna().sum()

In [ ]:
# Deletando dados faltantes:
data.dropna(inplace=True)

In [ ]:
# Missing values também podem estar representadas através do número 0.
# Por conta disso, deve-se verificar se os zeros são missing values ou se são inerentes ao atributo:
num_zeros = (data == 0).sum()
print('Quantidade de valores nulos:\n', num_zeros)

#### • Embaralhando os dados:

In [ ]:
data = shuffle(data, random_state=1)

#### • Reduzindo a escala dos dados:
Essa etapa garante que os dados estejam entre 0 e 1.

In [ ]:
# Cria uma array a partir dos dados:
z = data.values

# Atribuindo nova escala os dados:
min_max_scaler = preprocessing.MinMaxScaler()
data_normalized = min_max_scaler.fit_transform(z)

In [ ]:
print("Dados normalizados:\n", data_normalized)

#### • Delimitação de entrada e saída:

In [ ]:
# Delimitando entrada e saída:
inputs_all = data_normalized[:, :-1]
targets_all = data_normalized[:, -1]

# Atribuindo valor ao número de atributos (número de colunas dos dados de entrada):
n_inputs = inputs_all.shape[1]

In [ ]:
# Exibindo valores em tela:
print("Inputs:\n", inputs_all)
print("Outputs:\n", targets_all)
print("Nº de atributos:\n", n_inputs)

#### • Divisão do conjunto de dados em treinamento e teste

In [ ]:
# Definindo o número total de amostras:
samples_count = data_normalized.shape[0]

# Definindo o número de amostras em cada subconjunto, supondo que desejemos uma distribuição 80-20 de treinamento e teste:
train_samples_count = int(0.8 * samples_count)
test_samples_count = int(0.2 * samples_count)

# Criando variáveis que registram as entradas e saídas para treinamento:
train_inputs = inputs_all[:train_samples_count]
train_targets = targets_all[:train_samples_count]

# Criando variáveis que registram as entradas e targets para teste:
test_inputs = inputs_all[train_samples_count:]
test_targets = targets_all[train_samples_count:]

## Criação dos Modelos
<p style='text-align: justify;'>Os modelos são do tipo SFNN e treinados de acordo com o algoritmo de retropropagação (ARP); exceto o Método 3, que utiliza o ELM.</p>

## Definindo função para cálculo de MSE
MSE é a função de perda, utilizado para a otimização do ARP.

In [ ]:
def mse(actual, pred):
    actual, pred = np.array(actual), np.array(pred)
    return np.square(np.subtract(actual, pred)).mean()

## MÉTODO 1 - WILCAR
Modelo SFNN, treinado via ARP, considerando os pesos iniciais (e bias) do primeiro neurônio oculto obtidos a partir de Linearização e Análise via Série de Taylor e pesos dos demais neurônios ocultos, bem como do neurônio de saída obtidos a partir de uma inicialização randômica a partir do Método de Xavier, preservando os pesos dos neurônios já treinados e utilizando-os como estimativas iniciais para o próximo treinamento.

### Aplicação de Regressão Linear no conjunto de dados
<p style='text-align: justify;'>A regressão é feita utilizando o módulo statsmodels.api. O método utilizado para a regressão é o Método dos Mínimos Quadrados.</p>

In [ ]:
# Adiciona a coluna das constantes
inputs_all_cte = sm.add_constant(inputs_all, prepend=True)
# Cria e ajusta o modelo
res = sm.OLS(targets_all, inputs_all_cte).fit()
# Imprime os resultados
print(res.summary())

### Resolução do sistema de equações proposto para obter os parâmetros iniciais da rede
<p style='text-align: justify;'>A resolução é feita utilizando o módulo scipy.optimize.fsolve. O módulo retorna as raízes das equações (não lineares) definidas por f(x)= 0, dada uma estimativa inicial.</p>

In [ ]:
# Definindo constantes:
x_M = []
for i in range(n_inputs):
    x_M.append(np.mean(inputs_all[:, i]))
k = []
for i in range(n_inputs):
    k.append(res.params[i+1])
b = res.params[0]

In [ ]:
# Definindo chutes iniciais para beta e wi como valores aleatórios entre 0 e 1:
x0 = np.random.rand(n_inputs+1, 1)

In [ ]:
# Definindo função sigmoide e sua derivada:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def dsigmoid(x):
    z = sigmoid(x)
    return z * (1 - z)

In [ ]:
# Inicializando variáveis wi e beta:
w = [0 for i in range(n_inputs)]
beta = 0

In [ ]:
# Definindo sistema de equações a ser resolvido:
def f(x):
    w, beta = x[:-1], x[-1]
    f = [sigmoid(np.dot(w, x_M)+beta)-(sigmoid(np.dot(w, x_M)+beta) -
                                       (sigmoid(np.dot(w, x_M)+beta))**2)*np.dot(w, x_M)-b]
    for i in range(n_inputs):
        f.append((sigmoid(np.dot(w, x_M)+beta) -
                 (sigmoid(np.dot(w, x_M)+beta))**2)*w[i]-k[i])
    return f

In [ ]:
# Resolvendo o sistema e imprimindo resultados (pesos e bias iniciais):
result = fsolve(f, x0)
print(result)

### Criando Método 1:

In [ ]:
class Method1:

    def __init__(self, x, y, n):
        # vetor de saídas preditas. Inicialmente um vetor de zeros.
        self.Yh = np.zeros((1, y.shape[0]))
        self.n = n  # número de neurônios ocultos.
        # número de neurônios em cada camada (entrada, intermediária e saída).
        self.dims = [x.shape[0], n, 1]
        # um dicionário que conterá os parâmetros W e b entre as camadas da rede (entrada-intermediária e intermediária-saída); ou seja: W1, b1, W2 e b2.
        self.param = {}
        self.ch = {}  # parâmetro intermediário de cálculo.
        self.grad = {}  # parâmetro intermediário de cálculo.
        self.loss = None  # erro de treinamento (MSE).
        self.lr = 0.1  # taxa de aprendizagem.
        # número de observações do conjunto de treinamento.
        self.sam = y.shape[0]

    def nInit(self):
        self.param['W1'] = np.random.uniform(-math.sqrt(6/(self.dims[0] + self.dims[1])), math.sqrt(
            6/(self.dims[0] + self.dims[1])), (self.dims[1], self.dims[0]))
        self.param['b1'] = np.zeros((self.dims[1], 1))
        self.param['W2'] = np.random.uniform(-math.sqrt(6/(self.dims[1] + self.dims[2])), math.sqrt(
            6/(self.dims[1] + self.dims[2])), (self.dims[2], self.dims[1]))
        self.param['b2'] = np.zeros((self.dims[2], 1))
        self.param['W1'][0] = result[:-1]
        self.param['b1'][0] = result[-1]
        return

    def forward(self, x):
        Z1 = self.param['W1'].dot(x) + self.param['b1']
        A1 = sigmoid(Z1)
        self.ch['Z1'], self.ch['A1'] = Z1, A1
        Z2 = self.param['W2'].dot(A1) + self.param['b2']
        A2 = sigmoid(Z2)
        self.ch['Z2'], self.ch['A2'] = Z2, A2
        self.Yh = A2
        return self.Yh

    def backward(self, x, y):
        dLoss_Yh = - (y-self.Yh[0])

        dLoss_Z2 = dLoss_Yh * dsigmoid(self.ch['Z2'])
        dLoss_A1 = np.dot(self.param["W2"].T, dLoss_Z2)
        dLoss_W2 = 1/self.ch['A1'].shape[1] * np.dot(dLoss_Z2, self.ch['A1'].T)
        dLoss_b2 = 1/self.ch['A1'].shape[1] * \
            np.dot(dLoss_Z2, np.ones([dLoss_Z2.shape[1], 1]))

        dLoss_Z1 = dLoss_A1 * dsigmoid(self.ch['Z1'])
        dLoss_A0 = np.dot(self.param["W1"].T, dLoss_Z1)
        dLoss_W1 = 1/x.shape[1] * np.dot(dLoss_Z1, x.T)
        dLoss_b1 = 1/x.shape[1] * \
            np.dot(dLoss_Z1, np.ones([dLoss_Z1.shape[1], 1]))

        self.param["W1"] = self.param["W1"] - self.lr * dLoss_W1
        self.param["b1"] = self.param["b1"] - self.lr * dLoss_b1
        self.param["W2"] = self.param["W2"] - self.lr * dLoss_W2
        self.param["b2"] = self.param["b2"] - self.lr * dLoss_b2

    def train(self, x, y, iter=1501):
        for i in range(0, iter):
            self.forward(x)
            self.backward(x, y)
        self.loss = mse(y, self.Yh[0])
        return

    def test(self, x, y):
        self.forward(x)
        return mse(y, self.Yh[0])

### Executando Método 1:

#### Inicializando rede com 1 neurônio oculto

In [ ]:
# Inicializando contagem do tempo:
ini = time.time()

# Abrindo listas para MSE e R2 (treino e teste):
mse_train_1 = []
mse_test_1 = []
r2_train_1 = []
r2_test_1 = []

# Abrindo lista para registro de pesos dos modelos:
params_models_1 = []

# Sinalização, em tela, início de rede com 1 neurônio oculto:
print("Iniciando rede com 1 neurônio oculto:")

# Definindo entrada e saída de treino:
x = train_inputs.T
y = train_targets.T

# Criando modelo:
Modelo1 = Method1(x, y, 1)

# Inicializando pesos:
Modelo1.nInit()

# Treinando modelo:
Modelo1.train(x, y)

# MSE de treino:
mse_train_1.append(Modelo1.loss)
print("MSE de treino:", Modelo1.loss)

# R2 de treino:
r2_train_1.append(r2_score(y, Modelo1.Yh[0]))
print("R2 de treino:", r2_score(y, Modelo1.Yh[0]))

# Registrando pesos já aprendidos:
params_models_1.append(Modelo1.param)

# Definindo entrada e saída de teste:
x = test_inputs.T
y = test_targets.T

# Testando modelo e calculando MSE de teste:
mse_test_1.append(Modelo1.test(x, y))

# R2 de teste:
r2_test_1.append(r2_score(y, Modelo1.Yh[0]))
print("R2 de teste:", r2_score(y, Modelo1.Yh[0]))

#### Expandindo rede até 250 neurônios ocultos

In [ ]:
# Inicializando loop para treinamento de rede com demais neurônios:
for n in range(2, 251):
    # Sinalização, em tela, de início de rede com n neurônios ocultos:
    print("Iniciando rede com", n, "neurônios ocultos:")

    # Definindo entrada e saída de treino:
    x = train_inputs.T
    y = train_targets.T

    # Criando Modelo:
    Modelo1 = Method1(x, y, n)

    # Inicializando novos pesos e utilizando pesos do treinamento anterior:
    Modelo1.nInit()
    Modelo1.param['W1'][:-1] = params_models_1[-1]['W1']
    Modelo1.param['b1'][:-1] = params_models_1[-1]['b1']
    Modelo1.param['W2'][-1][:-1] = params_models_1[-1]['W2']
    Modelo1.param['b2'] = params_models_1[-1]['b2']

    # Treinando modelo:
    Modelo1.train(x, y)

    # Registro e exibição de MSE de treino:
    mse_train_1.append(Modelo1.loss)
    print("MSE de treino:", Modelo1.loss)

    # Registro e exibição de R2 de treino:
    r2_train_1.append(r2_score(y, Modelo1.Yh[0]))
    print("R2 de treino:", r2_score(y, Modelo1.Yh[0]))

    # Registrando pesos já aprendidos:
    params_models_1.append(Modelo1.param)

    # Definindo entrada e saída de teste:
    x = test_inputs.T
    y = test_targets.T

    # Testando modelo e calculando MSE de teste:
    mse_test_1.append(Modelo1.test(x, y))

    # Registro e exibição de R2 de teste:
    r2_test_1.append(r2_score(y, Modelo1.Yh[0]))
    print("R2 de teste:", r2_score(y, Modelo1.Yh[0]))

### Análise da performance dos modelos gerados
Métricas de análise:
- Tempo de processamento (s): tempo necessário para a criação, treinamento e teste dos modelos;
- Função de Perda (MSE de Treino);
- R2 de Treino;
- MSE de Teste;
- R2 de Teste.

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Tempo de processamento (s):", fim-ini)

In [ ]:
# Criação e exibição de Gráfico do MSE de treino:
plt.plot(mse_train_1)
plt.ylabel('Train MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['WILCA-R'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de treino:
plt.plot(r2_train_1)
plt.ylabel('Train R2')
plt.xlabel('Hidden Nodes')
plt.legend(['WILCA-R'])
plt.show()

# Exibindo em tela R2 de treino associado ao melhor modelo gerado:
print("R2 de treino associado ao modelo com R2 máximo de teste:",
      r2_train_1[r2_test_1.index(max(r2_test_1))])

In [ ]:
# Criação e exibição de Gráfico do MSE de teste:
plt.plot(mse_test_1)
plt.ylabel('Test MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['WILCA-R'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de teste:
plt.plot(r2_test_1)
plt.ylabel('Test R2')
plt.xlabel('Hidden Nodes')
plt.legend(['WILCA-R'])
plt.show()

# Exibindo, em tela, R2 máximo de teste:
print("R2 máximo de teste obtido:", max(r2_test_1))

# Exibindo, em tela, número ótimo de neurônios ocultos:
print("Número ótimo de neurônios ocultos:", (r2_test_1.index(max(r2_test_1)))+1)

### Análise de Ganhos
O cálculo dos ganhos é realizado tendo em vista o melhor modelo gerado a partir do método.

In [ ]:
# Inicializando modelo com quantidade ótima de neurônios ocultos
Modelo1 = Method1(x, y, r2_test_1.index(max(r2_test_1))+1)

# Atribuindo pesos ótimos calculados previamente:
Modelo1.nInit()
Modelo1.param['W1'] = params_models_1[r2_test_1.index(max(r2_test_1))]['W1']
Modelo1.param['b1'] = params_models_1[r2_test_1.index(max(r2_test_1))]['b1']
Modelo1.param['W2'] = params_models_1[r2_test_1.index(max(r2_test_1))]['W2']
Modelo1.param['b2'] = params_models_1[r2_test_1.index(max(r2_test_1))]['b2']

In [ ]:
# Registrando vetor base de input e valor de output associado (trata-se da primeira linha do conjunto de teste)
x = test_inputs.T
y = test_targets.T
x_base = []
for k in range(x.shape[0]):
    x_base.append(x[k][0])
x_base = np.asarray(x_base)
x_base = x_base.reshape(x.shape[0], 1)
y_base = Modelo1.forward(x_base)[0]

In [ ]:
# Registrando ganhos para o método 1, promovendo variação de 0.1 em cada input, individualmente
k1 = []
for k in range(len(x_base)):
    x_var = x_base.copy()
    x_var[k] = x_var[k]+0.1
    new_y = Modelo1.forward(x_var)[0]
    k1.append((new_y - y_base) / (x_var[k] - x_base[k]))

In [ ]:
k1

## MÉTODO 2 - RIXM
Modelo SFNN considerando inicialização de pesos a partir de uma inicialização randômica (Método de Xavier) e bias inicializados com valor nulo.

### Criando Método 2:

In [ ]:
class Method2:

    def __init__(self, x, y, n):
        # vetor de saídas preditas. Inicialmente um vetor de zeros.
        self.Yh = np.zeros((1, y.shape[0]))
        self.n = n  # número de neurônios ocultos.
        # número de neurônios em cada camada (entrada, intermediária e saída).
        self.dims = [x.shape[0], n, 1]
        # um dicionário que conterá os parâmetros W e b entre as camadas da rede (entrada-intermediária e intermediária-saída); ou seja: W1, b1, W2 e b2.
        self.param = {}
        self.ch = {}  # parâmetro intermediário de cálculo.
        self.grad = {}  # parâmetro intermediário de cálculo.
        self.loss = None  # erro de treinamento (MSE).
        self.lr = 0.1  # taxa de aprendizagem.
        # número de observações do conjunto de treinamento.
        self.sam = y.shape[0]

    def nInit(self):
        self.param['W1'] = np.random.uniform(-math.sqrt(6/(self.dims[0] + self.dims[1])), math.sqrt(
            6/(self.dims[0] + self.dims[1])), (self.dims[1], self.dims[0]))
        self.param['b1'] = np.zeros((self.dims[1], 1))
        self.param['W2'] = np.random.uniform(-math.sqrt(6/(self.dims[1] + self.dims[2])), math.sqrt(
            6/(self.dims[1] + self.dims[2])), (self.dims[2], self.dims[1]))
        self.param['b2'] = np.zeros((self.dims[2], 1))
        return

    def forward(self, x):
        Z1 = self.param['W1'].dot(x) + self.param['b1']
        A1 = sigmoid(Z1)
        self.ch['Z1'], self.ch['A1'] = Z1, A1
        Z2 = self.param['W2'].dot(A1) + self.param['b2']
        A2 = sigmoid(Z2)
        self.ch['Z2'], self.ch['A2'] = Z2, A2
        self.Yh = A2
        return self.Yh

    def backward(self,x,y):
        dLoss_Yh = - (y-self.Yh[0])

        dLoss_Z2 = dLoss_Yh * dsigmoid(self.ch['Z2'])
        dLoss_A1 = np.dot(self.param["W2"].T, dLoss_Z2)
        dLoss_W2 = 1/self.ch['A1'].shape[1] * np.dot(dLoss_Z2, self.ch['A1'].T)
        dLoss_b2 = 1/self.ch['A1'].shape[1] * \
            np.dot(dLoss_Z2, np.ones([dLoss_Z2.shape[1], 1]))

        dLoss_Z1 = dLoss_A1 * dsigmoid(self.ch['Z1'])
        dLoss_A0 = np.dot(self.param["W1"].T, dLoss_Z1)
        dLoss_W1 = 1/x.shape[1] * np.dot(dLoss_Z1, x.T)
        dLoss_b1 = 1/x.shape[1] * \
            np.dot(dLoss_Z1, np.ones([dLoss_Z1.shape[1], 1]))

        self.param["W1"] = self.param["W1"] - self.lr * dLoss_W1
        self.param["b1"] = self.param["b1"] - self.lr * dLoss_b1
        self.param["W2"] = self.param["W2"] - self.lr * dLoss_W2
        self.param["b2"] = self.param["b2"] - self.lr * dLoss_b2

    def train(self, x, y, iter=1501):
        for i in range(0, iter):
            self.forward(x)
            self.backward(x,y)
        self.loss = mse(y, self.Yh[0])
        return

    def test(self, x, y):
        self.forward(x)
        return mse(y, self.Yh[0])

### Executando Método 2:

In [ ]:
# Inicializando contagem do tempo:
ini = time.time()

# Abrindo listas para MSE e R2 (treino e teste):
mse_train_2 = []
mse_test_2 = []
r2_train_2 = []
r2_test_2 = []

# Abrindo lista para registro de pesos dos modelos:
params_models_2 = []

# Inicializando loop para neurônios ocultos:
for n in range(1, 251):
    # Sinalização em tela de início de rede com n neurônios ocultos:
    print("Iniciando rede com", n, "neurônios ocultos:")

    # Definindo entrada e saída de treino:
    x = train_inputs.T
    y = train_targets.T

    # Criando modelo com n neurônios ocultos:
    Modelo2 = Method2(x, y, n)

    # Inicializando pesos:
    Modelo2.nInit()

    # Treinando modelo:
    Modelo2.train(x, y)

    # Registrando e exibindo MSE de treino:
    mse_train_2.append(Modelo2.loss)
    print("MSE de treino:", Modelo2.loss)

    # Registrando e exibindo R2 de treino:
    r2_train_2.append(r2_score(y, Modelo2.Yh[0]))
    print("R2 de treino:", r2_score(y, Modelo2.Yh[0]))

    # Registrando pesos já aprendidos:
    params_models_2.append(Modelo2.param)

    # Definindo entrada e saída de teste:
    x = test_inputs.T
    y = test_targets.T

    # Testando modelo e registrando MSE de teste:
    mse_test_2.append(Modelo2.test(x, y))

    # Registrando e exibindo R2 de teste:
    r2_test_2.append(r2_score(y, Modelo2.Yh[0]))
    print("R2 de teste:", r2_score(y, Modelo2.Yh[0]))

### Análise da performance dos modelos gerados
Métricas de análise:
- Tempo de processamento (s): tempo necessário para a criação, treinamento e teste dos modelos;
- Função de Perda (MSE de Treino);
- R2 de Treino;
- MSE de Teste;
- R2 de Teste.

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Tempo de processamento:", fim-ini)

In [ ]:
# Criação e exibição de Gráfico do MSE de treino:
plt.plot(mse_train_2)
plt.ylabel('Train MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de treino:
plt.plot(r2_train_2)
plt.ylabel('Train R2')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

# Exibindo em tela R2 de treino associado ao melhor modelo gerado:
print("R2 de treino associado ao modelo com R2 máximo de teste:",
      r2_train_2[r2_test_2.index(max(r2_test_2))])

In [ ]:
# Criação e exibição de Gráfico do MSE de teste:
plt.plot(mse_train_2)
plt.ylabel('Test MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
plt.show()

In [ ]:
# Gráfico do R2 de teste:
plt.plot(r2_test_2)
plt.ylabel('Test R2')
plt.xlabel('Hidden Nodes')
plt.legend(['RIXM'])
# plt.ylim(-1,1)
plt.show()

# Exibindo, em tela, R2 máximo de teste:
print("R2 máximo de teste obtido:", max(r2_test_2))

# Exibindo, em tela, número ótimo de neurônios ocultos:
print("Número ótimo de neurônios ocultos:", (r2_test_2.index(max(r2_test_2)))+1)

### Análise de Ganhos
O cálculo dos ganhos é realizado tendo em vista o melhor modelo gerado a partir do método.

In [ ]:
# Inicializando modelo com quantidade ótima de neurônios ocultos
Modelo2 = Method2(x, y, r2_test_2.index(max(r2_test_2))+1)

# Atribuindo pesos ótimos calculados previamente:
Modelo2.nInit()
Modelo2.param['W1'] = params_models_2[r2_test_2.index(max(r2_test_2))]['W1']
Modelo2.param['b1'] = params_models_2[r2_test_2.index(max(r2_test_2))]['b1']
Modelo2.param['W2'] = params_models_2[r2_test_2.index(max(r2_test_2))]['W2']
Modelo2.param['b2'] = params_models_2[r2_test_2.index(max(r2_test_2))]['b2']

In [ ]:
# Registrando vetor base de input e valor de output associado (trata-se da primeira linha do conjunto de teste)
x = test_inputs.T
y = test_targets.T
x_base = []
for k in range(x.shape[0]):
    x_base.append(x[k][0])
x_base = np.asarray(x_base)
x_base = x_base.reshape(x.shape[0], 1)
y_base = Modelo2.forward(x_base)[0]

In [ ]:
# Registrando ganhos para o método 2, promovendo variação de 0.1 em cada input, individualmente
k2 = []
for k in range(len(x_base)):
    x_var = x_base.copy()
    x_var[k] = x_var[k]+0.1
    new_y = Modelo2.forward(x_var)[0]
    k2.append((new_y - y_base) / (x_var[k] - x_base[k]))

In [ ]:
k2

## MÉTODO 3 - ELM
Modelo ELM considerando os pesos iniciais dos neurônio oculto obtidos a partir de uma distribuição uniforme dentro do intervalo de -1 a 1; biases são inicializados com valor nulo.

### Definindo função de ativação ReLU:

In [ ]:
def relu(x):
    return np.maximum(x, 0, x)

### Criando Método 3:

In [ ]:
def hidden_nodes(X, input_weights, biases):
    G = np.dot(X, input_weights)
    G = G + biases
    H = relu(G)
    return H


def predict_test(X, output_weights, input_weights, biases):
    out = hidden_nodes(X, input_weights, biases)
    out = np.dot(out, output_weights)
    return out

### Executando Método 3:

In [ ]:
# Inicializando contagem de tempo:
ini = time.time()

# Abrindo listas para R2 (treino e teste):
mse_train_3 = []
mse_test_3 = []
r2_train_3 = []
r2_test_3 = []

# Abrindo lista para registro de pesos dos modelos:
params_models_3 = []

# Definindo tamanho do conjunto de entrada:
input_size = train_inputs.shape[1]

# Inicializando loop para neurônios ocultos:
for n in range(1, 251):
    # Treino:
    input_weights = np.random.uniform(-1, 1, (input_size, n))
    biases = np.zeros(n)
    print("Iniciando rede com", n, "neurônios ocultos")
    out = hidden_nodes(train_inputs, input_weights, biases)
    output_weights = np.dot(pinv(out), train_targets)
    out = np.dot(out, output_weights)
    actual = train_targets
    mse_train_3.append(mse(actual, out))
    r2_train_3.append(r2_score(actual, out))
    # Registrando pesos já aprendidos:
    params_models_3.append(output_weights)

    # Teste:
    test_predict = predict_test(test_inputs, output_weights, input_weights, biases)
    actual = test_targets
    mse_test_3.append(mse(actual, test_predict))
    r2_test_3.append(r2_score(actual, test_predict))

### Análise da performance dos modelos gerados
Métricas de análise:
- Tempo de processamento (s): tempo necessário para a criação, treinamento e teste dos modelos;
- Função de Perda (MSE de Treino);
- R2 de Treino;
- MSE de Teste;
- R2 de Teste.

In [ ]:
# Finalizando e exibindo contagem de tempo de processamento:
fim = time.time()
print("Tempo de processamento:", fim-ini)

In [ ]:
# Criação e exibição de Gráfico do MSE de treino:
plt.plot(mse_train_3)
plt.ylabel('Train MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do MSE de treino com zoom:
plt.plot(mse_train_3)
plt.ylabel('Train MSE')
plt.xlabel('Hidden Nodes')
plt.xlim(0, 50)
plt.legend(['ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de treino associado ao melhor modelo gerado:
plt.plot(r2_train_3)
plt.ylabel('Train R2')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()
# Exibindo em tela R2 de treino:
print("R2 de treino associado ao melhor modelo gerado:",
      r2_train_3[r2_test_3.index(max(r2_test_3))])

In [ ]:
# Criação e exibição de Gráfico do MSE de teste:
plt.plot(mse_test_3)
plt.ylabel('Test MSE')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()

In [ ]:
# Criação e exibição de Gráfico do R2 de teste:
plt.plot(r2_test_3)
plt.ylabel('Test R2')
plt.xlabel('Hidden Nodes')
plt.legend(['ELM'])
plt.show()
print("R2 máximo de teste obtido:", max(r2_test_3))
print("Número ótimo de neurônios ocultos:", (r2_test_3.index(max(r2_test_3)))+1)

In [ ]:
# Criação e exibição de Gráfico do R2 de teste com zoom:
plt.plot(r2_test_3)
plt.ylabel('Test R2')
plt.xlabel('Hidden Nodes')
plt.xlim(0, 50)
plt.legend(['ELM'])
plt.show()

### Análise de Ganhos
O cálculo dos ganhos é realizado tendo em vista o melhor modelo gerado a partir do método.

In [ ]:
# Atribuindo pesos ótimos calculados previamente associado à quantidade ótima de neurônios ocultos:
output_weights = params_models_3[r2_test_3.index(max(r2_test_3))]
input_weights = np.random.uniform(-1, 1, (input_size, r2_test_3.index(max(r2_test_3))+1))
biases = np.zeros(r2_test_3.index(max(r2_test_3))+1)

In [ ]:
# Registrando vetor base de input e valor de output associado (trata-se da primeira linha do conjunto de teste)
x = test_inputs.T
y = test_targets.T
x_base = []
for k in range(x.shape[0]):
    x_base.append(x[k][0])
y_base = predict_test(x_base, output_weights, input_weights, biases)

In [ ]:
# Registrando ganhos para o método 3, promovendo variação de 0.1 em cada input, individualmente
k3 = []
for k in range(len(x_base)):
    x_var = x_base.copy()
    x_var[k] = x_var[k]+0.1
    new_y = predict_test(x_var, output_weights, input_weights, biases)
    k3.append((new_y - y_base) / (x_var[k] - x_base[k]))

In [ ]:
k3

## SALVANDO GANHOS DOS TRÊS MODELOS FINAIS

In [ ]:
k = pd.DataFrame([k1, k2, k3]).T

In [ ]:
k.to_csv('Gains_'+filename+'.csv', header=False,
         sep=';', index=False, float_format="%.3f")

## FIM DO CÓDIGO